# CNN Model with MFCC Maps

This notebook trains a simple convolutional neural network for binary audio deepfake detection.

Labels:

- `0 = bona-fide`
- `1 = synthetic/deepfake`

MFCCs are kept as coefficient-by-time maps with one channel, so the CNN can learn local spectro-temporal patterns.


## 1. Environment Setup

Run this notebook from a clean Google Colab session or locally.


## 2. Package Installation


This cell checks whether the libraries required by the CNN workflow are installed and installs only the missing packages before the remaining audio-processing, analysis, plotting, and modelling steps run.

In [1]:
# Purpose: Checks whether the libraries required by the CNN workflow are installed and installs
# only the missing packages before the remaining audio-processing, analysis, plotting, and
# modelling steps run.
import importlib.util
import subprocess
import sys

# List the import names and their corresponding installable package names.
required_packages = [
    ("librosa", "librosa"),
    ("soundfile", "soundfile"),
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("sklearn", "scikit-learn"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("joblib", "joblib"),
    ("tensorflow", "tensorflow"),
]

# Install only dependencies that are not already available in the active kernel.
missing = [pip_name for import_name, pip_name in required_packages if importlib.util.find_spec(import_name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages are already installed.")


Required packages are already installed.


## 3. Imports


This cell imports the standard-library and third-party tools used by the CNN workflow and provides a print-based fallback when the richer notebook display function is unavailable.

In [2]:
# Purpose: Imports the standard-library and third-party tools used by the CNN workflow and
# provides a print-based fallback when the richer notebook display function is unavailable.
import json
import os
import random
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Activation, Conv2D, Dense, Dropout, Flatten, Input, MaxPooling2D
from tensorflow.keras.models import Sequential

try:
    from IPython.display import display
except Exception:
    display = print


## 4. Configuration and Random Seeds


This cell fixes the available random seeds for reproducibility and defines the shared audio, MFCC, class, sampling, and CNN training settings used later in the notebook.

In [3]:
# Purpose: Fixes the available random seeds for reproducibility and defines the shared audio,
# MFCC, class, sampling, and CNN training settings used later in the notebook.
# Use a fixed seed so sampling, splitting, and model initialisation can be repeated.
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

# Configure the fixed audio representation and MFCC analysis window.
SAMPLE_RATE = 22050
FIXED_DURATION_SECONDS = 5.0
N_MFCC = 40
N_FFT = 1024
HOP_LENGTH = int(round(SAMPLE_RATE * 0.010))
WIN_LENGTH = int(round(SAMPLE_RATE * 0.025))

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg", ".m4a"}
CLASS_NAMES = {0: "bona_fide", 1: "synthetic"}

# Keep dataset and training limits together so quick runs are easy to configure.
MAX_FILES_PER_CLASS = None  # Set to a small number, such as 40, for a quick check.
EPOCHS = 30
BATCH_SIZE = 128


## 5. Dataset Paths


This cell defines and calls a portable helper that mounts Google Drive when Colab is available and otherwise continues without failing in a local environment.

In [4]:
# Purpose: Defines and calls a portable helper that mounts Google Drive when Colab is available
# and otherwise continues without failing in a local environment.
# Mount Google Drive when the notebook is running in Colab.
def mount_drive_if_colab(mount_point="/content/drive"):
    try:
        from google.colab import drive
        drive.mount(mount_point)
    except Exception:
        print("Not running in Colab, or Google Drive is already available.")

mount_drive_if_colab()


Mounted at /content/drive


This cell defines the synthetic and bona-fide dataset paths, creates the output folders required by the CNN workflow, and prints the active locations for confirmation.

In [5]:
# Purpose: Defines the synthetic and bona-fide dataset paths, creates the output folders
# required by the CNN workflow, and prints the active locations for confirmation.
# Change these paths to match your Google Drive dataset folders.
# Set the shared root used to locate the two audio classes.
PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")
SYNTHETIC_AUDIO_DIR = PROJECT_ROOT / "Datasets" / "MLAAD_10pct"
BONA_FIDE_AUDIO_DIR = PROJECT_ROOT / "Datasets" / "M_AILABS_bona_fide_subset"

# Build a model-specific output hierarchy under the project root.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cnn"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"
MODELS_DIR = OUTPUT_DIR / "models"
TABLES_DIR = OUTPUT_DIR / "tables"

# Create every output directory before any files are saved.
for directory in [OUTPUT_DIR, FIGURES_DIR, METRICS_DIR, MODELS_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Synthetic audio directory:", SYNTHETIC_AUDIO_DIR)
print("Bona-fide audio directory:", BONA_FIDE_AUDIO_DIR)
print("Output directory:", OUTPUT_DIR)


Synthetic audio directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Datasets/MLAAD_10pct
Bona-fide audio directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/Datasets/M_AILABS_bona_fide_subset
Output directory: /content/drive/MyDrive/Colab Notebooks/Education/INM701/outputs/cnn


## 6. Dataset Loading / Audio File Scan


This cell defines a recursive scanner that validates a dataset folder and records each supported audio file's path, class label, inferred language, and size in a Pandas table.

In [6]:
# Purpose: Defines a recursive scanner that validates a dataset folder and records each
# supported audio file's path, class label, inferred language, and size in a Pandas table.
# Convert the supported files under one class directory into manifest rows.
def scan_audio_files(root_dir, label, class_name):
    root_dir = Path(root_dir)
    if not root_dir.exists():
        raise FileNotFoundError(f"Folder not found: {root_dir}")

    rows = []
    for path in sorted(root_dir.rglob("*")):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            rows.append({
                "path": str(path),
                "relative_path": str(path.relative_to(root_dir)),
                "label": label,
                "class_name": class_name,
                "language": path.parent.name,
                "file_size_mb": path.stat().st_size / (1024 * 1024),
            })
    return pd.DataFrame(rows)


This cell scans both class directories, optionally limits each class for a quick run, combines and reproducibly shuffles the records into one manifest, and displays a short preview.

In [7]:
# Purpose: Scans both class directories, optionally limits each class for a quick run, combines
# and reproducibly shuffles the records into one manifest, and displays a short preview.
# Scan the synthetic and bona-fide directories separately with their correct labels.
synthetic_manifest = scan_audio_files(SYNTHETIC_AUDIO_DIR, label=1, class_name=CLASS_NAMES[1])
bona_fide_manifest = scan_audio_files(BONA_FIDE_AUDIO_DIR, label=0, class_name=CLASS_NAMES[0])

# Optionally draw a reproducible smaller sample for a quick test run.
if MAX_FILES_PER_CLASS:
    synthetic_manifest = synthetic_manifest.sample(min(MAX_FILES_PER_CLASS, len(synthetic_manifest)), random_state=RANDOM_STATE)
    bona_fide_manifest = bona_fide_manifest.sample(min(MAX_FILES_PER_CLASS, len(bona_fide_manifest)), random_state=RANDOM_STATE)

# Merge both classes into one shuffled manifest for later splitting.
manifest = pd.concat([bona_fide_manifest, synthetic_manifest], ignore_index=True)
manifest = manifest.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

print("Total files:", len(manifest))
display(manifest.head())


Total files: 12944


,path,relative_path,label,class_name,language,file_size_mb
0,/content/drive/MyDrive/Colab Notebooks/Educati...,fake/en/Llasa-1B-Multilingual/pink_fairy_book_...,1,synthetic,Llasa-1B-Multilingual,0.368462
1,/content/drive/MyDrive/Colab Notebooks/Educati...,fake/de/Chatterbox Multilingual/altehaus_009_1...,1,synthetic,Chatterbox Multilingual,0.415565
2,/content/drive/MyDrive/Colab Notebooks/Educati...,fake/en/Metavoice-1B/northandsouth_38_f000272.wav,1,synthetic,Metavoice-1B,0.055557
3,/content/drive/MyDrive/Colab Notebooks/Educati...,en/en/northandsouth_05_f000169.wav,0,bona_fide,en,0.130991
4,/content/drive/MyDrive/Colab Notebooks/Educati...,en/en/poisoned_pen_08_f000016.wav,0,bona_fide,en,0.302805


## 7. Dataset Inspection


This cell summarises the manifest by class and language and displays the overall class counts together with the 20 most frequent class-language combinations.

In [8]:
# Purpose: Summarises the manifest by class and language and displays the overall class counts
# together with the 20 most frequent class-language combinations.
# Count recordings by class and by class-language combination.
class_summary = manifest.groupby(["label", "class_name"]).size().reset_index(name="files")
language_summary = manifest.groupby(["class_name", "language"]).size().reset_index(name="files")

display(class_summary)
display(language_summary.sort_values("files", ascending=False).head(20))


,label,class_name,files
0,0,bona_fide,3997
1,1,synthetic,8947


,class_name,language,files
1,bona_fide,en,3282
0,bona_fide,de,715
6,synthetic,Edge-TTS,407
43,synthetic,OmniVoice,402
55,synthetic,VoxCPM2,295
9,synthetic,Fish-S2-Pro,288
56,synthetic,Voxtral,283
18,synthetic,KugelAudio,214
20,synthetic,LEMAS-TTS,195
15,synthetic,Kani-TTS-370M,191


## 8. Data Quality Checks


This cell measures missing paths, files absent from disk, duplicate paths, and represented classes, then stops the workflow if both labels are not present or duplicate recordings could compromise the experiment.

In [9]:
# Purpose: Measures missing paths, files absent from disk, duplicate paths, and represented
# classes, then stops the workflow if both labels are not present or duplicate recordings could
# compromise the experiment.
# Calculate manifest checks before any expensive audio processing begins.
quality_checks = pd.DataFrame([
    {"check": "missing_paths", "value": int(manifest["path"].isna().sum())},
    {"check": "missing_files", "value": int((~manifest["path"].map(lambda p: Path(p).exists())).sum())},
    {"check": "duplicate_paths", "value": int(manifest["path"].duplicated().sum())},
    {"check": "classes_present", "value": ", ".join(map(str, sorted(manifest["label"].unique())))},
])

display(quality_checks)

# Stop early if the manifest does not contain both required target classes.
if set(manifest["label"].unique()) != {0, 1}:
    raise ValueError("Both classes are required: 0=bona-fide and 1=synthetic/deepfake.")
if manifest["path"].duplicated().any():
    raise ValueError("Duplicate audio paths were found. Remove duplicates before training.")


,check,value
0,missing_paths,0
1,missing_files,0
2,duplicate_paths,0
3,classes_present,"0, 1"


## 9. Audio Preprocessing

Audio is loaded as mono, resampled, padded or truncated, and normalised.


## 10. MFCC Feature Extraction


This cell computes the expected number of time frames, standardises each recording to a fixed duration and amplitude, and extracts a consistently sized two-dimensional MFCC map for CNN input.

In [10]:
# Purpose: Computes the expected number of time frames, standardises each recording to a fixed
# duration and amplitude, and extracts a consistently sized two-dimensional MFCC map for CNN
# input.
# Derive the fixed MFCC time dimension from the audio and framing settings.
EXPECTED_FRAMES = 1 + max(0, int(SAMPLE_RATE * FIXED_DURATION_SECONDS) - N_FFT) // HOP_LENGTH
print("Expected CNN MFCC map shape:", (N_MFCC, EXPECTED_FRAMES, 1))


# Resample, pad or truncate, and peak-normalise every waveform consistently.
def load_audio_fixed(path):
    audio, _ = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    target_length = int(SAMPLE_RATE * FIXED_DURATION_SECONDS)

    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    return audio.astype(np.float32)


# Transform the standardised waveform into the MFCC representation required downstream.
def extract_mfcc_map(path):
    audio = load_audio_fixed(path)
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=SAMPLE_RATE,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH,
        center=False,
    ).astype(np.float32)

    if mfcc.shape[1] < EXPECTED_FRAMES:
        mfcc = np.pad(mfcc, ((0, 0), (0, EXPECTED_FRAMES - mfcc.shape[1])))
    else:
        mfcc = mfcc[:, :EXPECTED_FRAMES]

    return mfcc


Expected CNN MFCC map shape: (40, 497, 1)


## 11. Label Preparation


This cell states the binary label mapping and displays the number of bona-fide and synthetic recordings so the target variable can be checked before splitting and modelling.

In [11]:
# Purpose: States the binary label mapping and displays the number of bona-fide and synthetic
# recordings so the target variable can be checked before splitting and modelling.
print("Label meaning: 0 = bona-fide, 1 = synthetic/deepfake")
display(manifest["label"].value_counts().sort_index().rename(index=CLASS_NAMES).to_frame("files"))


Label meaning: 0 = bona-fide, 1 = synthetic/deepfake


,files
label,
bona_fide,3997
synthetic,8947


## 12. Train / Validation / Test Split


This cell creates reproducible stratified training, validation, and test partitions in a 70:15:15 ratio, resets their row indexes, and displays the size and class balance of each split.

In [12]:
# Purpose: Creates reproducible stratified training, validation, and test partitions in a
# 70:15:15 ratio, resets their row indexes, and displays the size and class balance of each
# split.
# Reserve 30 percent of the manifest before dividing it equally into validation and test data.
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=manifest["label"],
)

# Split the held-out portion equally while preserving the class distribution.
validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(validation_df), len(test_df)],
    "bona_fide": [int((train_df["label"] == 0).sum()), int((validation_df["label"] == 0).sum()), int((test_df["label"] == 0).sum())],
    "synthetic": [int((train_df["label"] == 1).sum()), int((validation_df["label"] == 1).sum()), int((test_df["label"] == 1).sum())],
})
display(split_summary)


,split,rows,bona_fide,synthetic
0,train,9060,2798,6262
1,validation,1942,599,1343
2,test,1942,600,1342


## 13. CNN MFCC Map Preparation


This cell computes the expected number of time frames, standardises each recording to a fixed duration and amplitude, and extracts a consistently sized two-dimensional MFCC map for CNN input.

In [13]:
# Purpose: Computes the expected number of time frames, standardises each recording to a fixed
# duration and amplitude, and extracts a consistently sized two-dimensional MFCC map for CNN
# input.
# Convert a manifest split into aligned model inputs, labels, and retained metadata.
def build_mfcc_map_table(df, split_name):
    maps = []
    labels = []
    kept_rows = []

    for _, row in df.iterrows():
        try:
            maps.append(extract_mfcc_map(row["path"]))
            labels.append(int(row["label"]))
            kept_rows.append(row)
        except Exception as exc:
            print(f"Skipping {row['path']}: {exc}")

    X = np.stack(maps).astype(np.float32)
    X = X[..., np.newaxis]
    y = np.array(labels, dtype=np.int64)
    meta = pd.DataFrame(kept_rows).reset_index(drop=True)
    print(f"{split_name}: {X.shape}")
    return X, y, meta

X_train, y_train, train_meta = build_mfcc_map_table(train_df, "train")
X_validation, y_validation, validation_meta = build_mfcc_map_table(validation_df, "validation")


train: (9060, 40, 497, 1)
validation: (1942, 40, 497, 1)


## 14. Train-only MFCC Map Standardisation


This cell calculates MFCC-map normalisation statistics from the training split only and applies them to both the training and validation tensors before reporting the CNN input shape.

In [14]:
# Purpose: Calculates MFCC-map normalisation statistics from the training split only and applies
# them to both the training and validation tensors before reporting the CNN input shape.
# Estimate normalisation statistics from the training tensor only.
mfcc_mean = X_train.mean(axis=(0, 2), keepdims=True)
mfcc_std = X_train.std(axis=(0, 2), keepdims=True) + 1e-6

X_train_scaled = (X_train - mfcc_mean) / mfcc_std
X_validation_scaled = (X_validation - mfcc_mean) / mfcc_std

print("MFCC map standardisation fitted on training data only.")
print("CNN input shape:", X_train_scaled.shape[1:])


MFCC map standardisation fitted on training data only.
CNN input shape: (40, 497, 1)


## 15. CNN Model Definition


This cell constructs and summarises an example CNN with two convolution and pooling stages, dropout regularisation, a dense hidden layer, and a sigmoid output for binary classification.

In [15]:
# Purpose: Constructs and summarises an example CNN with two convolution and pooling stages,
# dropout regularisation, a dense hidden layer, and a sigmoid output for binary classification.
# Construct a representative estimator so its architecture or configuration can be inspected.
example_model = Sequential()
example_model.add(Input(shape=X_train_scaled.shape[1:]))
example_model.add(Conv2D(32, kernel_size=(3, 3), strides=1, padding="same"))
example_model.add(Activation("relu"))
example_model.add(MaxPooling2D(pool_size=(2, 2)))
example_model.add(Dropout(0.30))
example_model.add(Conv2D(64, kernel_size=(3, 3), strides=1, padding="same"))
example_model.add(Activation("relu"))
example_model.add(MaxPooling2D(pool_size=(2, 2)))
example_model.add(Dropout(0.30))
example_model.add(Flatten())
example_model.add(Dense(64, activation="relu"))
example_model.add(Dense(1, activation="sigmoid"))

example_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
example_model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 40, 497, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 40, 497, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 20, 248, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 248, 32)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 20, 248, 64)    │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 20, 248, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 124, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 10, 124, 64)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 79360)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     5,079,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,097,985 (19.45 MB)

 Trainable params: 5,097,985 (19.45 MB)

 Non-trainable params: 0 (0.00 B)

## 16. Training


This cell documents that model fitting happens in the following validation experiment and that early stopping uses validation loss while restoring the best recorded epoch.

In [16]:
# Purpose: Documents that model fitting happens in the following validation experiment and that
# early stopping uses validation loss while restoring the best recorded epoch.
# The model is trained in the validation experiment loop below.
# Early stopping watches validation loss and restores the best validation epoch.


## 17. Validation Hyperparameter Experiments


This cell trains the candidate CNN architectures with early stopping and best-checkpoint saving, measures their validation performance, ranks them by F1 score, and reloads the selected model.

In [17]:
# Purpose: Trains the candidate CNN architectures with early stopping and best-checkpoint
# saving, measures their validation performance, ranks them by F1 score, and reloads the
# selected model.
# Define the candidate configurations that will be compared on validation data.
experiments = [
    {"name": "cnn_32_64", "filters": [32, 64], "dropout": 0.30, "learning_rate": 0.001},
    {"name": "cnn_16_32", "filters": [16, 32], "dropout": 0.25, "learning_rate": 0.001},
]

validation_results = []
histories = {}
model_paths = {}

# Train and evaluate each candidate without using the test split for selection.
for experiment in experiments:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(RANDOM_STATE)

    model = Sequential()
    model.add(Input(shape=X_train_scaled.shape[1:]))

    for filters in experiment["filters"]:
        model.add(Conv2D(filters, kernel_size=(3, 3), strides=1, padding="same"))
        model.add(Activation("relu"))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(experiment["dropout"]))

    model.add(Flatten())
    model.add(Dense(64, activation="relu"))
    model.add(Dropout(experiment["dropout"]))
    model.add(Dense(1, activation="sigmoid"))

    optimizer = tf.keras.optimizers.Adam(learning_rate=experiment["learning_rate"])
    model.compile(loss="binary_crossentropy", optimizer=optimizer, metrics=["accuracy"])

    model_path = MODELS_DIR / f"{experiment['name']}.keras"
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(model_path), monitor="val_loss", save_best_only=True),
    ]

    history = model.fit(
        X_train_scaled,
        y_train,
        validation_data=(X_validation_scaled, y_validation),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2,
    )

    validation_probability = model.predict(X_validation_scaled, batch_size=BATCH_SIZE).ravel()
    validation_pred = (validation_probability >= 0.5).astype(int)
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)

    validation_results.append({
        "model": "CNN",
        "variant": experiment["name"],
        "filters": str(experiment["filters"]),
        "dropout": experiment["dropout"],
        "learning_rate": experiment["learning_rate"],
        "accuracy": accuracy_score(y_validation, validation_pred),
        "precision": precision_score(y_validation, validation_pred, zero_division=0),
        "recall": recall_score(y_validation, validation_pred, zero_division=0),
        "f1": f1_score(y_validation, validation_pred, zero_division=0),
        "best_epoch": best_epoch,
        "epochs_trained": len(history.history["loss"]),
        "early_stopped": len(history.history["loss"]) < EPOCHS,
        "best_val_loss": float(np.min(history.history["val_loss"])),
    })

    histories[experiment["name"]] = history.history
    model_paths[experiment["name"]] = model_path

# Rank candidate models by validation F1 score before selecting the winner.
validation_results_df = pd.DataFrame(validation_results).sort_values("f1", ascending=False)
display(validation_results_df)

best_variant = validation_results_df.iloc[0]["variant"]
best_model = tf.keras.models.load_model(str(model_paths[best_variant]))
print("Selected from validation data:", best_variant)


Epoch 1/30
71/71 - 21s - 295ms/step - accuracy: 0.8953 - loss: 0.5064 - val_accuracy: 0.9614 - val_loss: 0.1703
Epoch 2/30
71/71 - 4s - 56ms/step - accuracy: 0.9635 - loss: 0.1453 - val_accuracy: 0.9737 - val_loss: 0.1113
Epoch 3/30
71/71 - 4s - 56ms/step - accuracy: 0.9732 - loss: 0.1004 - val_accuracy: 0.9737 - val_loss: 0.0941
Epoch 4/30
71/71 - 4s - 56ms/step - accuracy: 0.9769 - loss: 0.0757 - val_accuracy: 0.9804 - val_loss: 0.0728
Epoch 5/30
71/71 - 4s - 58ms/step - accuracy: 0.9810 - loss: 0.0602 - val_accuracy: 0.9784 - val_loss: 0.0651
Epoch 6/30
71/71 - 4s - 56ms/step - accuracy: 0.9861 - loss: 0.0435 - val_accuracy: 0.9779 - val_loss: 0.0648
Epoch 7/30
71/71 - 4s - 56ms/step - accuracy: 0.9886 - loss: 0.0344 - val_accuracy: 0.9825 - val_loss: 0.0581
Epoch 8/30
71/71 - 4s - 56ms/step - accuracy: 0.9906 - loss: 0.0303 - val_accuracy: 0.9835 - val_loss: 0.0551
Epoch 9/30
71/71 - 4s - 56ms/step - accuracy: 0.9921 - loss: 0.0225 - val_accuracy: 0.9851 - val_loss: 0.0523
Epoch 10

,model,variant,filters,dropout,learning_rate,accuracy,precision,recall,f1,best_epoch,epochs_trained,early_stopped,best_val_loss
0,CNN,cnn_32_64,"[32, 64]",0.30,0.001,0.985582,0.992509,0.986597,0.989544,10,15,True,0.048805
1,CNN,cnn_16_32,"[16, 32]",0.25,0.001,0.981462,0.987323,0.985853,0.986587,12,17,True,0.064560


Selected from validation data: cnn_32_64


## 18. Final Test Evaluation


This cell processes the untouched test split only after selecting the CNN variant, generates binary predictions, calculates the final accuracy, precision, recall, and F1 score, and displays the results.

In [ ]:
# Purpose: Processes the untouched test split only after selecting the CNN variant, generates
# binary predictions, calculates the final accuracy, precision, recall, and F1 score, and
# displays the results.
# The test split is processed only after choosing the model from validation results.
# Build the untouched test inputs only after validation-based model selection is complete.
X_test, y_test, test_meta = build_mfcc_map_table(test_df, "test")
X_test_scaled = (X_test - mfcc_mean) / mfcc_std

# Convert the selected model's outputs into positive-class probabilities.
test_probability = best_model.predict(X_test_scaled, batch_size=BATCH_SIZE).ravel()
test_pred = (test_probability >= 0.5).astype(int)

# Calculate the final held-out classification metrics from thresholded predictions.
test_results_df = pd.DataFrame([{
    "model": "CNN",
    "selected_variant": best_variant,
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1": f1_score(y_test, test_pred, zero_division=0),
}])

display(test_results_df)


## 19. Additional Subgroup / Multilingual Analysis


This cell joins the final predictions to the test metadata and calculates language-specific performance for subgroups containing at least five recordings, avoiding unstable summaries for very small groups.

In [ ]:
# Purpose: Joins the final predictions to the test metadata and calculates language-specific
# performance for subgroups containing at least five recordings, avoiding unstable summaries for
# very small groups.
# Prepare language-level metrics using the final test predictions.
subgroup_rows = []
subgroup_df = test_meta.copy().reset_index(drop=True)
subgroup_df["y_true"] = y_test
subgroup_df["y_pred"] = test_pred

# Skip very small language groups and score those with enough test examples.
for language, group in subgroup_df.groupby("language"):
    if len(group) < 5:
        continue
    subgroup_rows.append({
        "language": language,
        "files": len(group),
        "accuracy": accuracy_score(group["y_true"], group["y_pred"]),
        "precision": precision_score(group["y_true"], group["y_pred"], zero_division=0),
        "recall": recall_score(group["y_true"], group["y_pred"], zero_division=0),
        "f1": f1_score(group["y_true"], group["y_pred"], zero_division=0),
    })

subgroup_results_df = pd.DataFrame(subgroup_rows)
if subgroup_results_df.empty:
    print("No language subgroup has at least 5 test files.")
else:
    display(subgroup_results_df.sort_values("f1", ascending=False))


## 20. Confusion Matrix and Saved Outputs


This cell saves the CNN validation and test results, language metrics when available, confusion-matrix data and figures, preprocessing state, and selected model, then prints the main artifact paths.

In [ ]:
# Purpose: Saves the CNN validation and test results, language metrics when available,
# confusion-matrix data and figures, preprocessing state, and selected model, then prints the
# main artifact paths.
# Define consistent destinations for evaluation, plots, preprocessing state, and models.
validation_results_path = TABLES_DIR / "cnn_validation_experiments.csv"
test_metrics_path = METRICS_DIR / "cnn_test_metrics.json"
confusion_matrix_path = METRICS_DIR / "cnn_confusion_matrix.csv"
subgroup_metrics_path = METRICS_DIR / "cnn_language_subgroup_metrics.csv"
history_plot_path = FIGURES_DIR / "cnn_training_history.png"
confusion_plot_path = FIGURES_DIR / "cnn_confusion_matrix.png"
standardisation_path = MODELS_DIR / "cnn_mfcc_standardisation_stats.npz"
best_model_path = MODELS_DIR / "cnn_best_model.keras"

validation_results_df.to_csv(validation_results_path, index=False)
with open(test_metrics_path, "w", encoding="utf-8") as f:
    json.dump(test_results_df.iloc[0].to_dict(), f, indent=2)

# Build a labelled confusion matrix from the final test predictions.
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
pd.DataFrame(cm, index=[CLASS_NAMES[0], CLASS_NAMES[1]], columns=[CLASS_NAMES[0], CLASS_NAMES[1]]).to_csv(confusion_matrix_path)
if not subgroup_results_df.empty:
    subgroup_results_df.to_csv(subgroup_metrics_path, index=False)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]], yticklabels=[CLASS_NAMES[0], CLASS_NAMES[1]])
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("CNN confusion matrix")
plt.tight_layout()
plt.savefig(confusion_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# Plot the selected neural model's training and validation curves across epochs.
best_history = histories[best_variant]
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(best_history["loss"], label="training")
plt.plot(best_history["val_loss"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(best_history["accuracy"], label="training")
plt.plot(best_history["val_accuracy"], label="validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(history_plot_path, dpi=300, bbox_inches="tight")
plt.show()

np.savez(standardisation_path, mean=mfcc_mean, std=mfcc_std)
best_model.save(str(best_model_path))

print("Saved validation results:", validation_results_path)
print("Saved test metrics:", test_metrics_path)
print("Saved confusion matrix:", confusion_matrix_path)
print("Saved MFCC standardisation stats:", standardisation_path)
print("Saved best model:", best_model_path)


## 21. Reproducibility Checks


This cell displays a compact reproducibility record for the CNN workflow, including the stratified split, train-only preprocessing, test-set isolation, label mapping, dependency design, and random seed.

In [21]:
# Purpose: Displays a compact reproducibility record for the CNN workflow, including the
# stratified split, train-only preprocessing, test-set isolation, label mapping, dependency
# design, and random seed.
# Collect the main reproducibility and leakage-prevention properties for display.
checks = {
    "self_contained": True,
    "external_helper_file_required": False,
    "custom_helper_imports": False,
    "split": "70/15/15 stratified by class label",
    "scaler_fitted_on": "training split only",
    "test_used_for_model_selection": False,
    "label_mapping": CLASS_NAMES,
    "random_state": RANDOM_STATE,
}

display(pd.DataFrame(list(checks.items()), columns=["check", "value"]))


,check,value
0,self_contained,True
1,external_helper_file_required,False
2,custom_helper_imports,False
3,split,70/15/15 stratified by class label
4,scaler_fitted_on,training split only
5,test_used_for_model_selection,False
6,label_mapping,"{0: 'bona_fide', 1: 'synthetic'}"
7,random_state,42
